In [ ]:
code = 'SREW'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output\\'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def SREW(bt, start_time, end_time, orderside, sl, intra_sl, om):
    try:
        start_dt = datetime.datetime.combine(bt.current_week_dates[0], start_time)
        end_dt = datetime.datetime.combine(bt.current_week_dates[-1], end_time)

        ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
        if ce_scrip is None: return None

        entry_time = start_dt
        std_open, std_high, std_low, std_close, std_sl_price, std_intra_sl_price, std_sl_flag, std_intra_sl_flag, _, std_sl_time, std_pnl = bt.sl_check_combine_leg(start_dt, end_dt, ce_scrip, pe_scrip, sl=sl, intra_sl=intra_sl, orderside=orderside, with_ohlc=True)
        
        first_straddle = [f"({ce_scrip}, {pe_scrip})", std_open, std_sl_flag, std_intra_sl_flag, std_sl_time, std_pnl]
        
        re_straddles = []
        for re_no in range(max_re):
            
            if std_sl_time and re_no < re_entries and (std_sl_time < end_dt - datetime.timedelta(minutes=5)):
                start_dt = std_sl_time
                ce_scrip, pe_scrip, ce_price, pe_price, _, start_dt = bt.get_strike(start_dt, end_dt, om=om)
                
                if ce_scrip is None:
                    std_sl_time = ''
                    re_straddles.extend(['', '', False, False, '', 0])
                    continue

                std_open, std_high, std_low, std_close, std_sl_price, std_intra_sl_price, std_sl_flag, std_intra_sl_flag, _, std_sl_time, std_pnl = bt.sl_check_combine_leg(start_dt, end_dt, ce_scrip, pe_scrip, sl=sl, intra_sl=intra_sl, orderside=orderside, with_ohlc=True)
                re_straddles.extend([f"({ce_scrip}, {pe_scrip})", std_open, std_sl_flag, std_intra_sl_flag, std_sl_time, std_pnl])
            else:
                re_straddles.extend(['', '', False, False, '', 0])

        return [code, bt.index, start_time, end_time, orderside, sl, intra_sl, om, bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), bt.from_dte, bt.to_dte, len(bt.current_week_dates), entry_time.time(), future_price] + first_straddle + re_straddles
    
    except Exception as e:
        print(e, [bt.index, bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), start_time, end_time, orderside, sl, intra_sl, om])
        return

In [ ]:
for row_idx in range(len(meta_data)):
    
    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, from_dte, to_dte, from_date, to_date, start_time, end_time, week_lists = get_meta_row_data(meta_row, pickle_path, weekly=True)
            max_re, re_entries = 10, 10

            log_cols = 'P_Strategy/P_Index/P_StartTime/P_EndTime/P_OrderSide/P_SL/P_intraSL/P_OM/Start.Date/End.Date/Start.DTE/End.DTE/DayCount/EntryTime/Future'

            for r in range(max_re+1):
                log_cols += f'/STD{r}.Scrip/STD{r}.Open/STD{r}.SL.Flag/STD{r}.IntraSL/STD{r}.SL.Time/STD{r}.PNL'
            log_cols = log_cols.split('/')

            for week_dates in week_lists:
                from_date = week_dates[0]
                to_date = week_dates[-1]

                file_name = f"{index} {week_dates[0].date()} {week_dates[-1].date()} {from_dte}-{to_dte} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    wbt = WeeklyBacktest(pickle_path, index, week_dates, from_dte, to_dte, start_time, end_time)

                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)

                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [SREW(wbt, row['entry_time'], row['exit_time'], row['orderside'], row['sl'], row['intra_sl'], row['om']) for idx, row in tqdm(chunk_parameter.iterrows(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)

                    del wbt
                    t2 = datetime.datetime.now()
                    print(t2-t1)

        except Exception as e:
            input(str(e))